In [1]:
import pandas as pd
import numpy as np
import pybaseball as pyb
import matplotlib.pyplot as plt

In [2]:
batting_stats = pyb.batting_stats(2025, qual=340)
print(batting_stats.columns.tolist())

['IDfg', 'Season', 'Name', 'Team', 'Age', 'G', 'AB', 'PA', 'H', '1B', '2B', '3B', 'HR', 'R', 'RBI', 'BB', 'IBB', 'SO', 'HBP', 'SF', 'SH', 'GDP', 'SB', 'CS', 'AVG', 'GB', 'FB', 'LD', 'IFFB', 'Pitches', 'Balls', 'Strikes', 'IFH', 'BU', 'BUH', 'BB%', 'K%', 'BB/K', 'OBP', 'SLG', 'OPS', 'ISO', 'BABIP', 'GB/FB', 'LD%', 'GB%', 'FB%', 'IFFB%', 'HR/FB', 'IFH%', 'BUH%', 'wOBA', 'wRAA', 'wRC', 'Bat', 'Fld', 'Rep', 'Pos', 'RAR', 'WAR', 'Dol', 'Spd', 'wRC+', 'WPA', '-WPA', '+WPA', 'RE24', 'REW', 'pLI', 'phLI', 'PH', 'WPA/LI', 'Clutch', 'FB% (Pitch)', 'FBv', 'SL%', 'SLv', 'CT%', 'CTv', 'CB%', 'CBv', 'CH%', 'CHv', 'SF%', 'SFv', 'KN%', 'KNv', 'XX%', 'PO%', 'wFB', 'wSL', 'wCT', 'wCB', 'wCH', 'wSF', 'wKN', 'wFB/C', 'wSL/C', 'wCT/C', 'wCB/C', 'wCH/C', 'wSF/C', 'wKN/C', 'O-Swing%', 'Z-Swing%', 'Swing%', 'O-Contact%', 'Z-Contact%', 'Contact%', 'Zone%', 'F-Strike%', 'SwStr%', 'BsR', 'FA% (sc)', 'FT% (sc)', 'FC% (sc)', 'FS% (sc)', 'FO% (sc)', 'SI% (sc)', 'SL% (sc)', 'CU% (sc)', 'KC% (sc)', 'EP% (sc)', 'CH% (

In [16]:
df = pd.read_csv('/Users/andredegregorio/hitter-cards/hitter_cards/data/clean2024.csv')
df.head()

,last_name,first_name,player_id,year,PA,wOBA,xwOBA,xBA,xSLG,Barrel%,...,O-Contact%,Z-Contact%,O-Swing%,Z-Swing%,Whiff%,Z-O Swing%,Pulled FB%,EV90,Sweet Spot%,Bat Speed
0,Judge,Aaron,592450.0,2024,704,0.476,0.480,0.310,0.724,0.269,...,0.489,0.807,0.213,0.719,0.121,0.506,20.512821,111.26,40.256410,75.681227
1,Jr.,Bobby,677951.0,2024,709,0.410,0.407,0.315,0.576,0.143,...,0.700,0.884,0.354,0.748,0.098,0.394,15.241636,109.43,36.245353,73.189638
2,Ohtani,Shohei,660271.0,2024,731,0.431,0.442,0.314,0.660,0.214,...,0.594,0.820,0.305,0.705,0.125,0.400,15.598291,111.93,37.179487,74.721365
3,Soto,Juan,665742.0,2024,713,0.421,0.463,0.317,0.647,0.197,...,0.674,0.864,0.213,0.601,0.074,0.388,11.930586,110.01,35.357918,72.939254
4,Henderson,Gunnar,683002.0,2024,719,0.381,0.373,0.283,0.492,0.112,...,0.623,0.858,0.262,0.686,0.098,0.424,10.736842,108.07,34.736842,74.725754


In [20]:
# Get Cal Raleigh's ID
player_lookup = pyb.playerid_lookup('Raleigh', 'Cal')
player_id = player_lookup['key_mlbam'].iloc[0]
print(f"Cal Raleigh MLB ID: {player_id}")

Cal Raleigh MLB ID: 663728


In [24]:
from utils import get_savant_data
from constants import BIP_EVENTS, SEASON_DATES

# Get 2024 season data
start_date = SEASON_DATES[2024]['REG_START']
end_date = SEASON_DATES[2024]['REG_END']

raw_data = get_savant_data(player_id, start_date, end_date)

In [ ]:
import pandas as pd
import numpy as np
import pybaseball as pyb
from utils import get_savant_data
from constants import BIP_EVENTS, SEASON_DATES

# Get Cal Raleigh's data
player_lookup = pyb.playerid_lookup('Raleigh', 'Cal')
player_id = player_lookup['key_mlbam'].iloc[0]

start_date = SEASON_DATES[2024]['REG_START']
end_date = SEASON_DATES[2024]['REG_END']

raw_data = get_savant_data(player_id, start_date, end_date)
bip_data = raw_data[raw_data['events'].isin(BIP_EVENTS)].copy()

print(f"Cal Raleigh 2024")
print(f"Total BIP: {len(bip_data)}\n")

# Calculate spray angle
bip_data['spray_angle'] = np.arctan(
    (bip_data['hc_x'] - 125.42) / (198.27 - bip_data['hc_y'])
) * 180 / np.pi * 0.75

# Adjust for batter side
bip_data['adj_spray_angle'] = np.where(
    bip_data['stand'] == 'L',
    -bip_data['spray_angle'],
    bip_data['spray_angle']
)

print("="*60)
print("VARIATION 1: Pull < -15°, Air = LA > 0°")
print("="*60)

# Variation 1
v1_pull = bip_data['adj_spray_angle'] < -15
v1_air = bip_data['launch_angle'] > 0
v1_gb = (bip_data['launch_angle'] >= -90) & (bip_data['launch_angle'] < 10)
v1_ld = (bip_data['launch_angle'] >= 10) & (bip_data['launch_angle'] < 25)
v1_fb = (bip_data['launch_angle'] >= 25) & (bip_data['launch_angle'] < 50)
v1_pu = bip_data['launch_angle'] >= 50

print(f"GB%: {v1_gb.mean() * 100:.1f}%")
print(f"AIR%: {v1_air.mean() * 100:.1f}%")
print(f"FB%: {v1_fb.mean() * 100:.1f}%")
print(f"LD%: {v1_ld.mean() * 100:.1f}%")
print(f"PU%: {v1_pu.mean() * 100:.1f}%")
print(f"Pull%: {v1_pull.mean() * 100:.1f}%")
print(f"Pull AIR%: {(v1_pull & v1_air).mean() * 100:.1f}%")

print("\n" + "="*60)
print("VARIATION 2: Pull < -15°, Air = LA >= 10°")
print("="*60)

# Variation 2
v2_pull = bip_data['adj_spray_angle'] < -15
v2_air = bip_data['launch_angle'] >= 10
v2_gb = bip_data['launch_angle'] < 10
v2_ld = (bip_data['launch_angle'] >= 10) & (bip_data['launch_angle'] < 25)
v2_fb = (bip_data['launch_angle'] >= 25) & (bip_data['launch_angle'] < 50)
v2_pu = bip_data['launch_angle'] >= 50

print(f"GB%: {v2_gb.mean() * 100:.1f}%")
print(f"AIR%: {v2_air.mean() * 100:.1f}%")
print(f"FB%: {v2_fb.mean() * 100:.1f}%")
print(f"LD%: {v2_ld.mean() * 100:.1f}%")
print(f"PU%: {v2_pu.mean() * 100:.1f}%")
print(f"Pull%: {v2_pull.mean() * 100:.1f}%")
print(f"Pull AIR%: {(v2_pull & v2_air).mean() * 100:.1f}%")

print("\n" + "="*60)
print("VARIATION 3: Pull < -17°, Air = LA > 0°")
print("="*60)

# Variation 3
v3_pull = bip_data['adj_spray_angle'] < -17
v3_air = bip_data['launch_angle'] > 0
v3_gb = (bip_data['launch_angle'] >= -90) & (bip_data['launch_angle'] < 10)
v3_ld = (bip_data['launch_angle'] >= 10) & (bip_data['launch_angle'] < 25)
v3_fb = (bip_data['launch_angle'] >= 25) & (bip_data['launch_angle'] < 50)
v3_pu = bip_data['launch_angle'] >= 50

print(f"GB%: {v3_gb.mean() * 100:.1f}%")
print(f"AIR%: {v3_air.mean() * 100:.1f}%")
print(f"FB%: {v3_fb.mean() * 100:.1f}%")
print(f"LD%: {v3_ld.mean() * 100:.1f}%")
print(f"PU%: {v3_pu.mean() * 100:.1f}%")
print(f"Pull%: {v3_pull.mean() * 100:.1f}%")
print(f"Pull AIR%: {(v3_pull & v3_air).mean() * 100:.1f}%")

print("\n" + "="*60)
print("VARIATION 4: Pull < -17°, Air = LA >= 10°")
print("="*60)

# Variation 4
v4_pull = bip_data['adj_spray_angle'] < -17
v4_air = bip_data['launch_angle'] >= 10
v4_gb = bip_data['launch_angle'] < 10
v4_ld = (bip_data['launch_angle'] >= 10) & (bip_data['launch_angle'] < 25)
v4_fb = (bip_data['launch_angle'] >= 25) & (bip_data['launch_angle'] < 50)
v4_pu = bip_data['launch_angle'] >= 50

print(f"GB%: {v4_gb.mean() * 100:.1f}%")
print(f"AIR%: {v4_air.mean() * 100:.1f}%")
print(f"FB%: {v4_fb.mean() * 100:.1f}%")
print(f"LD%: {v4_ld.mean() * 100:.1f}%")
print(f"PU%: {v4_pu.mean() * 100:.1f}%")
print(f"Pull%: {v4_pull.mean() * 100:.1f}%")
print(f"Pull AIR%: {(v4_pull & v4_air).mean() * 100:.1f}%")


print("\n" + "="*60)
print("SAVANT TARGET (2024):")
print("="*60)
print("GB%: 31.1%")
print("AIR%: 68.9%")
print("FB%: 36.7%")
print("LD%: 21.3%")
print("PU%: 10.9%")
print("Pull%: 45.5%")
print("Pull AIR%: 29.5%")

Cal Raleigh 2024
Total BIP: 376

VARIATION 1: Pull < -15°, Air = LA > 0°
GB%: 31.6%
AIR%: 79.5%
FB%: 29.3%
LD%: 21.8%
PU%: 17.0%
Pull%: 45.7%
Pull AIR%: 33.2%

VARIATION 2: Pull < -15°, Air = LA >= 10°
GB%: 31.6%
AIR%: 68.1%
FB%: 29.3%
LD%: 21.8%
PU%: 17.0%
Pull%: 45.7%
Pull AIR%: 26.3%

VARIATION 3: Pull < -17°, Air = LA > 0°
GB%: 31.6%
AIR%: 79.5%
FB%: 29.3%
LD%: 21.8%
PU%: 17.0%
Pull%: 42.6%
Pull AIR%: 31.1%

VARIATION 4: Pull < -17°, Air = LA >= 10°
GB%: 31.6%
AIR%: 68.1%
FB%: 29.3%
LD%: 21.8%
PU%: 17.0%
Pull%: 42.6%
Pull AIR%: 24.7%

SAVANT TARGET (2024):
GB%: 31.1%
AIR%: 68.9%
FB%: 36.7%
LD%: 21.3%
PU%: 10.9%
Pull%: 45.5%
Pull AIR%: 29.5%


In [42]:
import pandas as pd
import numpy as np
import pybaseball as pyb
from utils import get_savant_data
from constants import BIP_EVENTS, SEASON_DATES

# Get Cal Raleigh's data
player_lookup = pyb.playerid_lookup('Raleigh', 'Cal')
player_id = player_lookup['key_mlbam'].iloc[0]

start_date = SEASON_DATES[2024]['REG_START']
end_date = SEASON_DATES[2024]['REG_END']

raw_data = get_savant_data(player_id, start_date, end_date)
bip_data = raw_data[raw_data['events'].isin(BIP_EVENTS)].copy()

print(f"Cal Raleigh 2024")
print(f"Total BIP: {len(bip_data)}\n")

# Calculate spray angle
bip_data['spray_angle'] = np.arctan(
    (bip_data['hc_x'] - 125.42) / (198.27 - bip_data['hc_y'])
) * 180 / np.pi * 0.75

# Adjust for batter side
bip_data['adj_spray_angle'] = np.where(
    bip_data['stand'] == 'L',
    -bip_data['spray_angle'],
    bip_data['spray_angle']
)

print("="*60)
print("VARIATION 5: Pull < -15°, Air >= 10°, FB cutoff at 50°")
print("="*60)

v5_pull = bip_data['adj_spray_angle'] < -15
v5_gb = bip_data['launch_angle'] < 10
v5_ld = (bip_data['launch_angle'] >= 10) & (bip_data['launch_angle'] < 25)
v5_fb = (bip_data['launch_angle'] >= 25) & (bip_data['launch_angle'] <= 50)
v5_pu = bip_data['launch_angle'] > 50
v5_air = bip_data['launch_angle'] >= 10

print(f"GB%: {v5_gb.mean() * 100:.1f}%")
print(f"AIR%: {v5_air.mean() * 100:.1f}%")
print(f"FB%: {v5_fb.mean() * 100:.1f}%")
print(f"LD%: {v5_ld.mean() * 100:.1f}%")
print(f"PU%: {v5_pu.mean() * 100:.1f}%")
print(f"Pull%: {v5_pull.mean() * 100:.1f}%")
print(f"Pull AIR%: {(v5_pull & v5_air).mean() * 100:.1f}%")

print("\n" + "="*60)
print("VARIATION 6: Pull < -15°, Air >= 10°, FB cutoff at 55°")
print("="*60)

v6_pull = bip_data['adj_spray_angle'] < -15
v6_gb = bip_data['launch_angle'] < 10
v6_ld = (bip_data['launch_angle'] >= 10) & (bip_data['launch_angle'] < 25)
v6_fb = (bip_data['launch_angle'] >= 25) & (bip_data['launch_angle'] < 55)
v6_pu = bip_data['launch_angle'] >= 55
v6_air = bip_data['launch_angle'] >= 10

print(f"GB%: {v6_gb.mean() * 100:.1f}%")
print(f"AIR%: {v6_air.mean() * 100:.1f}%")
print(f"FB%: {v6_fb.mean() * 100:.1f}%")
print(f"LD%: {v6_ld.mean() * 100:.1f}%")
print(f"PU%: {v6_pu.mean() * 100:.1f}%")
print(f"Pull%: {v6_pull.mean() * 100:.1f}%")
print(f"Pull AIR%: {(v6_pull & v6_air).mean() * 100:.1f}%")

print("\n" + "="*60)
print("VARIATION 7: Pull < -15°, Air >= 10°, FB cutoff at 60°")
print("="*60)

v7_pull = bip_data['adj_spray_angle'] < -15
v7_gb = bip_data['launch_angle'] < 10
v7_ld = (bip_data['launch_angle'] >= 10) & (bip_data['launch_angle'] < 25)
v7_fb = (bip_data['launch_angle'] >= 25) & (bip_data['launch_angle'] < 60)
v7_pu = bip_data['launch_angle'] >= 60
v7_air = bip_data['launch_angle'] >= 10

print(f"GB%: {v7_gb.mean() * 100:.1f}%")
print(f"AIR%: {v7_air.mean() * 100:.1f}%")
print(f"FB%: {v7_fb.mean() * 100:.1f}%")
print(f"LD%: {v7_ld.mean() * 100:.1f}%")
print(f"PU%: {v7_pu.mean() * 100:.1f}%")
print(f"Pull%: {v7_pull.mean() * 100:.1f}%")
print(f"Pull AIR%: {(v7_pull & v7_air).mean() * 100:.1f}%")

print("\n" + "="*60)
print("VARIATION 8: Pull < -16°, Air >= 10°, FB cutoff at 50°")
print("="*60)

v8_pull = bip_data['adj_spray_angle'] < -16
v8_gb = bip_data['launch_angle'] < 10
v8_ld = (bip_data['launch_angle'] >= 10) & (bip_data['launch_angle'] < 25)
v8_fb = (bip_data['launch_angle'] >= 25) & (bip_data['launch_angle'] <= 50)
v8_pu = bip_data['launch_angle'] > 50
v8_air = bip_data['launch_angle'] >= 10

print(f"GB%: {v8_gb.mean() * 100:.1f}%")
print(f"AIR%: {v8_air.mean() * 100:.1f}%")
print(f"FB%: {v8_fb.mean() * 100:.1f}%")
print(f"LD%: {v8_ld.mean() * 100:.1f}%")
print(f"PU%: {v8_pu.mean() * 100:.1f}%")
print(f"Pull%: {v8_pull.mean() * 100:.1f}%")
print(f"Pull AIR%: {(v8_pull & v8_air).mean() * 100:.1f}%")

print("\n" + "="*60)
print("VARIATION 9: Pull < -16°, Air >= 10°, FB cutoff at 55°")
print("="*60)

v9_pull = bip_data['adj_spray_angle'] < -16
v9_gb = bip_data['launch_angle'] < 10
v9_ld = (bip_data['launch_angle'] >= 10) & (bip_data['launch_angle'] < 25)
v9_fb = (bip_data['launch_angle'] >= 25) & (bip_data['launch_angle'] < 55)
v9_pu = bip_data['launch_angle'] >= 55
v9_air = bip_data['launch_angle'] >= 10

print(f"GB%: {v9_gb.mean() * 100:.1f}%")
print(f"AIR%: {v9_air.mean() * 100:.1f}%")
print(f"FB%: {v9_fb.mean() * 100:.1f}%")
print(f"LD%: {v9_ld.mean() * 100:.1f}%")
print(f"PU%: {v9_pu.mean() * 100:.1f}%")
print(f"Pull%: {v9_pull.mean() * 100:.1f}%")
print(f"Pull AIR%: {(v9_pull & v9_air).mean() * 100:.1f}%")

print("\n" + "="*60)
print("SAVANT TARGET (2024):")
print("="*60)
print("GB%: 31.1%")
print("AIR%: 68.9%")
print("FB%: 36.7%")
print("LD%: 21.3%")
print("PU%: 10.9%")
print("Pull%: 45.5%")
print("Pull AIR%: 29.5%")

print("\n" + "="*60)
print("SUMMARY - Find the variation with:")
print("="*60)
print("- AIR% closest to 68.9%")
print("- FB% closest to 36.7%")
print("- PU% closest to 10.9%")
print("- Pull% closest to 45.5%")
print("- Pull AIR% closest to 29.5%")

Cal Raleigh 2024
Total BIP: 376

VARIATION 5: Pull < -15°, Air >= 10°, FB cutoff at 50°
GB%: 31.6%
AIR%: 68.1%
FB%: 29.8%
LD%: 21.8%
PU%: 16.5%
Pull%: 45.7%
Pull AIR%: 26.3%

VARIATION 6: Pull < -15°, Air >= 10°, FB cutoff at 55°
GB%: 31.6%
AIR%: 68.1%
FB%: 33.8%
LD%: 21.8%
PU%: 12.5%
Pull%: 45.7%
Pull AIR%: 26.3%

VARIATION 7: Pull < -15°, Air >= 10°, FB cutoff at 60°
GB%: 31.6%
AIR%: 68.1%
FB%: 36.7%
LD%: 21.8%
PU%: 9.6%
Pull%: 45.7%
Pull AIR%: 26.3%

VARIATION 8: Pull < -16°, Air >= 10°, FB cutoff at 50°
GB%: 31.6%
AIR%: 68.1%
FB%: 29.8%
LD%: 21.8%
PU%: 16.5%
Pull%: 43.9%
Pull AIR%: 25.5%

VARIATION 9: Pull < -16°, Air >= 10°, FB cutoff at 55°
GB%: 31.6%
AIR%: 68.1%
FB%: 33.8%
LD%: 21.8%
PU%: 12.5%
Pull%: 43.9%
Pull AIR%: 25.5%

SAVANT TARGET (2024):
GB%: 31.1%
AIR%: 68.9%
FB%: 36.7%
LD%: 21.3%
PU%: 10.9%
Pull%: 45.5%
Pull AIR%: 29.5%

SUMMARY - Find the variation with:
- AIR% closest to 68.9%
- FB% closest to 36.7%
- PU% closest to 10.9%
- Pull% closest to 45.5%
- Pull AIR% closest

In [43]:
import pandas as pd
import numpy as np
import pybaseball as pyb
from utils import get_savant_data
from constants import BIP_EVENTS, SEASON_DATES

# Get Cal Raleigh's data
player_lookup = pyb.playerid_lookup('Raleigh', 'Cal')
player_id = player_lookup['key_mlbam'].iloc[0]

start_date = SEASON_DATES[2024]['REG_START']
end_date = SEASON_DATES[2024]['REG_END']

raw_data = get_savant_data(player_id, start_date, end_date)
bip_data = raw_data[raw_data['events'].isin(BIP_EVENTS)].copy()

print(f"Cal Raleigh 2024")
print(f"Total BIP: {len(bip_data)}\n")

# Calculate spray angle
bip_data['spray_angle'] = np.arctan(
    (bip_data['hc_x'] - 125.42) / (198.27 - bip_data['hc_y'])
) * 180 / np.pi * 0.75

# Adjust for batter side
bip_data['adj_spray_angle'] = np.where(
    bip_data['stand'] == 'L',
    -bip_data['spray_angle'],
    bip_data['spray_angle']
)

print("="*60)
print("VARIATION 10: Pull <= -15°, Air >= 10°, FB < 60°")
print("="*60)

v10_pull = bip_data['adj_spray_angle'] <= -15
v10_gb = bip_data['launch_angle'] < 10
v10_ld = (bip_data['launch_angle'] >= 10) & (bip_data['launch_angle'] < 25)
v10_fb = (bip_data['launch_angle'] >= 25) & (bip_data['launch_angle'] < 60)
v10_pu = bip_data['launch_angle'] >= 60
v10_air = bip_data['launch_angle'] >= 10

print(f"GB%: {v10_gb.mean() * 100:.1f}%")
print(f"AIR%: {v10_air.mean() * 100:.1f}%")
print(f"FB%: {v10_fb.mean() * 100:.1f}%")
print(f"LD%: {v10_ld.mean() * 100:.1f}%")
print(f"PU%: {v10_pu.mean() * 100:.1f}%")
print(f"Pull%: {v10_pull.mean() * 100:.1f}%")
print(f"Pull AIR%: {(v10_pull & v10_air).mean() * 100:.1f}%")

print("\n" + "="*60)
print("VARIATION 11: Pull < -14°, Air >= 10°, FB < 60°")
print("="*60)

v11_pull = bip_data['adj_spray_angle'] < -14
v11_gb = bip_data['launch_angle'] < 10
v11_ld = (bip_data['launch_angle'] >= 10) & (bip_data['launch_angle'] < 25)
v11_fb = (bip_data['launch_angle'] >= 25) & (bip_data['launch_angle'] < 60)
v11_pu = bip_data['launch_angle'] >= 60
v11_air = bip_data['launch_angle'] >= 10

print(f"GB%: {v11_gb.mean() * 100:.1f}%")
print(f"AIR%: {v11_air.mean() * 100:.1f}%")
print(f"FB%: {v11_fb.mean() * 100:.1f}%")
print(f"LD%: {v11_ld.mean() * 100:.1f}%")
print(f"PU%: {v11_pu.mean() * 100:.1f}%")
print(f"Pull%: {v11_pull.mean() * 100:.1f}%")
print(f"Pull AIR%: {(v11_pull & v11_air).mean() * 100:.1f}%")

print("\n" + "="*60)
print("VARIATION 12: Pull < -15°, Air > 9°, FB < 60°")
print("="*60)

v12_pull = bip_data['adj_spray_angle'] < -15
v12_gb = bip_data['launch_angle'] <= 9
v12_ld = (bip_data['launch_angle'] > 9) & (bip_data['launch_angle'] < 25)
v12_fb = (bip_data['launch_angle'] >= 25) & (bip_data['launch_angle'] < 60)
v12_pu = bip_data['launch_angle'] >= 60
v12_air = bip_data['launch_angle'] > 9

print(f"GB%: {v12_gb.mean() * 100:.1f}%")
print(f"AIR%: {v12_air.mean() * 100:.1f}%")
print(f"FB%: {v12_fb.mean() * 100:.1f}%")
print(f"LD%: {v12_ld.mean() * 100:.1f}%")
print(f"PU%: {v12_pu.mean() * 100:.1f}%")
print(f"Pull%: {v12_pull.mean() * 100:.1f}%")
print(f"Pull AIR%: {(v12_pull & v12_air).mean() * 100:.1f}%")

print("\n" + "="*60)
print("VARIATION 13: Pull < -15°, Air >= 10°, FB <= 60°")
print("="*60)

v13_pull = bip_data['adj_spray_angle'] < -15
v13_gb = bip_data['launch_angle'] < 10
v13_ld = (bip_data['launch_angle'] >= 10) & (bip_data['launch_angle'] < 25)
v13_fb = (bip_data['launch_angle'] >= 25) & (bip_data['launch_angle'] <= 60)
v13_pu = bip_data['launch_angle'] > 60
v13_air = bip_data['launch_angle'] >= 10

print(f"GB%: {v13_gb.mean() * 100:.1f}%")
print(f"AIR%: {v13_air.mean() * 100:.1f}%")
print(f"FB%: {v13_fb.mean() * 100:.1f}%")
print(f"LD%: {v13_ld.mean() * 100:.1f}%")
print(f"PU%: {v13_pu.mean() * 100:.1f}%")
print(f"Pull%: {v13_pull.mean() * 100:.1f}%")
print(f"Pull AIR%: {(v13_pull & v13_air).mean() * 100:.1f}%")

print("\n" + "="*60)
print("VARIATION 14: Pull < -14°, Air > 9°, FB < 60°")
print("="*60)

v14_pull = bip_data['adj_spray_angle'] < -14
v14_gb = bip_data['launch_angle'] <= 9
v14_ld = (bip_data['launch_angle'] > 9) & (bip_data['launch_angle'] < 25)
v14_fb = (bip_data['launch_angle'] >= 25) & (bip_data['launch_angle'] < 60)
v14_pu = bip_data['launch_angle'] >= 60
v14_air = bip_data['launch_angle'] > 9

print(f"GB%: {v14_gb.mean() * 100:.1f}%")
print(f"AIR%: {v14_air.mean() * 100:.1f}%")
print(f"FB%: {v14_fb.mean() * 100:.1f}%")
print(f"LD%: {v14_ld.mean() * 100:.1f}%")
print(f"PU%: {v14_pu.mean() * 100:.1f}%")
print(f"Pull%: {v14_pull.mean() * 100:.1f}%")
print(f"Pull AIR%: {(v14_pull & v14_air).mean() * 100:.1f}%")

print("\n" + "="*60)
print("VARIATION 15: Pull < -15.5°, Air >= 10°, FB < 60°")
print("="*60)

v15_pull = bip_data['adj_spray_angle'] < -15.5
v15_gb = bip_data['launch_angle'] < 10
v15_ld = (bip_data['launch_angle'] >= 10) & (bip_data['launch_angle'] < 25)
v15_fb = (bip_data['launch_angle'] >= 25) & (bip_data['launch_angle'] < 60)
v15_pu = bip_data['launch_angle'] >= 60
v15_air = bip_data['launch_angle'] >= 10

print(f"GB%: {v15_gb.mean() * 100:.1f}%")
print(f"AIR%: {v15_air.mean() * 100:.1f}%")
print(f"FB%: {v15_fb.mean() * 100:.1f}%")
print(f"LD%: {v15_ld.mean() * 100:.1f}%")
print(f"PU%: {v15_pu.mean() * 100:.1f}%")
print(f"Pull%: {v15_pull.mean() * 100:.1f}%")
print(f"Pull AIR%: {(v15_pull & v15_air).mean() * 100:.1f}%")

print("\n" + "="*60)
print("SAVANT TARGET (2024):")
print("="*60)
print("GB%: 31.1%")
print("AIR%: 68.9%")
print("FB%: 36.7%")
print("LD%: 21.3%")
print("PU%: 10.9%")
print("Pull%: 45.5%")
print("Pull AIR%: 29.5%")

print("\n" + "="*60)
print("BEST IMPROVEMENTS:")
print("="*60)
print("V10: Pull <= -15° (includes -15) - should increase Pull AIR%")
print("V11: Pull < -14° (less strict) - should increase Pull AIR%")
print("V12: Air > 9° instead of >= 10° - should increase AIR%")
print("V13: FB <= 60° instead of < 60° - should increase FB%")
print("V14: Combo of -14° and >9°")
print("V15: Pull < -15.5° (middle ground)")

Cal Raleigh 2024
Total BIP: 376

VARIATION 10: Pull <= -15°, Air >= 10°, FB < 60°
GB%: 31.6%
AIR%: 68.1%
FB%: 36.7%
LD%: 21.8%
PU%: 9.6%
Pull%: 45.7%
Pull AIR%: 26.3%

VARIATION 11: Pull < -14°, Air >= 10°, FB < 60°
GB%: 31.6%
AIR%: 68.1%
FB%: 36.7%
LD%: 21.8%
PU%: 9.6%
Pull%: 48.1%
Pull AIR%: 27.7%

VARIATION 12: Pull < -15°, Air > 9°, FB < 60°
GB%: 31.6%
AIR%: 68.1%
FB%: 36.7%
LD%: 21.8%
PU%: 9.6%
Pull%: 45.7%
Pull AIR%: 26.3%

VARIATION 13: Pull < -15°, Air >= 10°, FB <= 60°
GB%: 31.6%
AIR%: 68.1%
FB%: 37.2%
LD%: 21.8%
PU%: 9.0%
Pull%: 45.7%
Pull AIR%: 26.3%

VARIATION 14: Pull < -14°, Air > 9°, FB < 60°
GB%: 31.6%
AIR%: 68.1%
FB%: 36.7%
LD%: 21.8%
PU%: 9.6%
Pull%: 48.1%
Pull AIR%: 27.7%

VARIATION 15: Pull < -15.5°, Air >= 10°, FB < 60°
GB%: 31.6%
AIR%: 68.1%
FB%: 36.7%
LD%: 21.8%
PU%: 9.6%
Pull%: 44.9%
Pull AIR%: 25.8%

SAVANT TARGET (2024):
GB%: 31.1%
AIR%: 68.9%
FB%: 36.7%
LD%: 21.3%
PU%: 10.9%
Pull%: 45.5%
Pull AIR%: 29.5%

BEST IMPROVEMENTS:
V10: Pull <= -15° (includes -15) - 

In [44]:
import pandas as pd
import numpy as np
import pybaseball as pyb
from utils import get_savant_data
from constants import BIP_EVENTS, SEASON_DATES

def test_player(last_name, first_name, season=2024):
    """Test Variation 7 formula on a player"""
    
    # Get player data
    player_lookup = pyb.playerid_lookup(last_name, first_name)
    player_id = player_lookup['key_mlbam'].iloc[0]
    
    start_date = SEASON_DATES[season]['REG_START']
    end_date = SEASON_DATES[season]['REG_END']
    
    raw_data = get_savant_data(player_id, start_date, end_date)
    bip_data = raw_data[raw_data['events'].isin(BIP_EVENTS)].copy()
    
    print(f"\n{'='*60}")
    print(f"{first_name} {last_name} {season}")
    print(f"{'='*60}")
    print(f"Total BIP: {len(bip_data)}")
    print(f"Bats: {bip_data['stand'].iloc[0]}")
    
    # Calculate spray angle
    bip_data['spray_angle'] = np.arctan(
        (bip_data['hc_x'] - 125.42) / (198.27 - bip_data['hc_y'])
    ) * 180 / np.pi * 0.75
    
    # Adjust for batter side
    bip_data['adj_spray_angle'] = np.where(
        bip_data['stand'] == 'L',
        -bip_data['spray_angle'],
        bip_data['spray_angle']
    )
    
    # VARIATION 7: Pull < -15°, Air >= 10°, FB < 60°
    pull = bip_data['adj_spray_angle'] < -15
    gb = bip_data['launch_angle'] < 10
    ld = (bip_data['launch_angle'] >= 10) & (bip_data['launch_angle'] < 25)
    fb = (bip_data['launch_angle'] >= 25) & (bip_data['launch_angle'] < 60)
    pu = bip_data['launch_angle'] >= 60
    air = bip_data['launch_angle'] >= 10
    
    print(f"\nVARIATION 7 Results:")
    print(f"GB%: {gb.mean() * 100:.1f}%")
    print(f"AIR%: {air.mean() * 100:.1f}%")
    print(f"FB%: {fb.mean() * 100:.1f}%")
    print(f"LD%: {ld.mean() * 100:.1f}%")
    print(f"PU%: {pu.mean() * 100:.1f}%")
    print(f"Pull%: {pull.mean() * 100:.1f}%")
    print(f"Pull AIR%: {(pull & air).mean() * 100:.1f}%")
    
    print(f"\n{'='*60}")
    print("PASTE SAVANT STATS BELOW FOR COMPARISON:")
    print("GB% | AIR% | FB% | LD% | PU% | Pull% | Pull AIR%")
    print(f"{'='*60}")

# Test Judge
test_player('Judge', 'Aaron')

# Test Soto
test_player('Soto', 'Juan')

print("\n" + "="*60)
print("INSTRUCTIONS:")
print("="*60)
print("Go to Baseball Savant for each player")
print("Find their 2024 batted ball breakdown table")
print("Paste the stats here so we can compare!")


Aaron Judge 2024
Total BIP: 390
Bats: R

VARIATION 7 Results:
GB%: 31.3%
AIR%: 68.2%
FB%: 39.7%
LD%: 24.9%
PU%: 3.6%
Pull%: 36.4%
Pull AIR%: 19.0%

PASTE SAVANT STATS BELOW FOR COMPARISON:
GB% | AIR% | FB% | LD% | PU% | Pull% | Pull AIR%

Juan Soto 2024
Total BIP: 461
Bats: L

VARIATION 7 Results:
GB%: 45.8%
AIR%: 54.0%
FB%: 26.0%
LD%: 22.6%
PU%: 5.4%
Pull%: 45.1%
Pull AIR%: 16.5%

PASTE SAVANT STATS BELOW FOR COMPARISON:
GB% | AIR% | FB% | LD% | PU% | Pull% | Pull AIR%

INSTRUCTIONS:
Go to Baseball Savant for each player
Find their 2024 batted ball breakdown table
Paste the stats here so we can compare!


In [46]:
import pandas as pd
import numpy as np
import pybaseball as pyb
from utils import get_savant_data
from constants import BIP_EVENTS, SEASON_DATES

def test_pull_thresholds(last_name, first_name, season=2024):
    """Test different pull angle thresholds to fix Pull AIR%"""
    
    # Get player data
    player_lookup = pyb.playerid_lookup(last_name, first_name)
    player_id = player_lookup['key_mlbam'].iloc[0]
    
    start_date = SEASON_DATES[season]['REG_START']
    end_date = SEASON_DATES[season]['REG_END']
    
    raw_data = get_savant_data(player_id, start_date, end_date)
    bip_data = raw_data[raw_data['events'].isin(BIP_EVENTS)].copy()
    
    print(f"\n{'='*60}")
    print(f"{first_name} {last_name} {season}")
    print(f"{'='*60}")
    
    # Calculate spray angle
    bip_data['spray_angle'] = np.arctan(
        (bip_data['hc_x'] - 125.42) / (198.27 - bip_data['hc_y'])
    ) * 180 / np.pi * 0.75
    
    # Adjust for batter side
    bip_data['adj_spray_angle'] = np.where(
        bip_data['stand'] == 'L',
        -bip_data['spray_angle'],
        bip_data['spray_angle']
    )
    
    # Fixed batted ball definitions (from Variation 7)
    gb = bip_data['launch_angle'] < 10
    ld = (bip_data['launch_angle'] >= 10) & (bip_data['launch_angle'] < 25)
    fb = (bip_data['launch_angle'] >= 25) & (bip_data['launch_angle'] < 60)
    pu = bip_data['launch_angle'] >= 60
    air = bip_data['launch_angle'] >= 10
    
    # Test different pull thresholds
    thresholds = [-12, -13, -14, -15, -16]
    
    print(f"\n{'Threshold':<12} {'Pull%':<10} {'Pull AIR%':<12}")
    print("-" * 35)
    
    for threshold in thresholds:
        pull = bip_data['adj_spray_angle'] < threshold
        pull_pct = pull.mean() * 100
        pull_air_pct = (pull & air).mean() * 100
        print(f"< {threshold}°{' '*(8-len(str(threshold)))} {pull_pct:>6.1f}%    {pull_air_pct:>6.1f}%")

# Test all three players
print("\n" + "="*60)
print("TESTING PULL ANGLE THRESHOLDS")
print("="*60)

test_pull_thresholds('Raleigh', 'Cal')
print("\nSAVANT TARGET: Pull%: 45.5% | Pull AIR%: 29.5%")

test_pull_thresholds('Judge', 'Aaron')
print("\nSAVANT TARGET: Pull%: 40.3% | Pull AIR%: 21.5%")

test_pull_thresholds('Soto', 'Juan')
print("\nSAVANT TARGET: Pull%: 42.3% | Pull AIR%: 19.1%")

print("\n" + "="*60)
print("ANALYSIS:")
print("="*60)
print("Find the threshold where:")
print("1. Pull% matches Savant (within 1-2%)")
print("2. Pull AIR% matches Savant (within 1-2%)")
print("If one threshold works across all 3 players, that's our answer!")


TESTING PULL ANGLE THRESHOLDS

Cal Raleigh 2024

Threshold    Pull%      Pull AIR%   
-----------------------------------
< -12°        51.1%      29.0%
< -13°        49.2%      28.2%
< -14°        48.1%      27.7%
< -15°        45.7%      26.3%
< -16°        43.9%      25.5%

SAVANT TARGET: Pull%: 45.5% | Pull AIR%: 29.5%

Aaron Judge 2024

Threshold    Pull%      Pull AIR%   
-----------------------------------
< -12°        39.7%      20.8%
< -13°        38.7%      20.3%
< -14°        37.7%      19.5%
< -15°        36.4%      19.0%
< -16°        34.6%      18.2%

SAVANT TARGET: Pull%: 40.3% | Pull AIR%: 21.5%

Juan Soto 2024

Threshold    Pull%      Pull AIR%   
-----------------------------------
< -12°        48.4%      18.2%
< -13°        46.4%      17.4%
< -14°        45.8%      16.7%
< -15°        45.1%      16.5%
< -16°        44.0%      15.8%

SAVANT TARGET: Pull%: 42.3% | Pull AIR%: 19.1%

ANALYSIS:
Find the threshold where:
1. Pull% matches Savant (within 1-2%)
2. Pull AIR

In [48]:
import pandas as pd
import numpy as np
import pybaseball as pyb
from utils import get_savant_data
from constants import BIP_EVENTS, SEASON_DATES

def test_variation_7(last_name, first_name, season=2024):
    """Test Variation 7 on a player and return results"""
    
    # Get player data
    player_lookup = pyb.playerid_lookup(last_name, first_name)
    player_id = player_lookup['key_mlbam'].iloc[0]
    
    start_date = SEASON_DATES[season]['REG_START']
    end_date = SEASON_DATES[season]['REG_END']
    
    raw_data = get_savant_data(player_id, start_date, end_date)
    bip_data = raw_data[raw_data['events'].isin(BIP_EVENTS)].copy()
    
    # Get batting side
    bat_side = bip_data['stand'].mode()[0] if len(bip_data) > 0 else 'N/A'
    
    # Calculate spray angle
    bip_data['spray_angle'] = np.arctan(
        (bip_data['hc_x'] - 125.42) / (198.27 - bip_data['hc_y'])
    ) * 180 / np.pi * 0.75
    
    # Adjust for batter side
    bip_data['adj_spray_angle'] = np.where(
        bip_data['stand'] == 'L',
        -bip_data['spray_angle'],
        bip_data['spray_angle']
    )
    
    # VARIATION 7
    pull = bip_data['adj_spray_angle'] < -15
    gb = bip_data['launch_angle'] < 10
    ld = (bip_data['launch_angle'] >= 10) & (bip_data['launch_angle'] < 25)
    fb = (bip_data['launch_angle'] >= 25) & (bip_data['launch_angle'] < 60)
    pu = bip_data['launch_angle'] >= 60
    air = bip_data['launch_angle'] >= 10
    
    print(f"\n{'='*60}")
    print(f"{first_name} {last_name} - Bats: {bat_side}")
    print(f"{'='*60}")
    print(f"GB%: {gb.mean() * 100:.1f}%")
    print(f"AIR%: {air.mean() * 100:.1f}%")
    print(f"FB%: {fb.mean() * 100:.1f}%")
    print(f"LD%: {ld.mean() * 100:.1f}%")
    print(f"PU%: {pu.mean() * 100:.1f}%")
    print(f"Pull%: {pull.mean() * 100:.1f}%")
    print(f"Pull AIR%: {(pull & air).mean() * 100:.1f}%")
    print(f"\nPaste Savant stats for comparison:")
    print(f"GB% | AIR% | FB% | LD% | PU% | Pull% | Pull AIR%")

# Test 5 more players - mix of lefties and righties
print("\n" + "="*60)
print("TESTING 5 MORE PLAYERS")
print("="*60)

# Test 5 more players - mix of lefties and righties
print("\n" + "="*60)
print("TESTING 5 MORE PLAYERS")
print("="*60)

# Righties
test_variation_7('Betts', 'Mookie')      # RHH
test_variation_7('Harper', 'Bryce')      # RHH (use instead of Alvarez)

# Lefties  
test_variation_7('Freeman', 'Freddie')   # LHH
test_variation_7('Schwarber', 'Kyle')    # LHH

# Switch
test_variation_7('Lindor', 'Francisco')  # Switch


print("\n" + "="*60)
print("GO FIND THESE PLAYERS ON SAVANT AND PASTE THEIR STATS")
print("="*60)


TESTING 5 MORE PLAYERS

TESTING 5 MORE PLAYERS

Mookie Betts - Bats: R
GB%: 28.1%
AIR%: 71.9%
FB%: 35.3%
LD%: 28.1%
PU%: 8.5%
Pull%: 30.2%
Pull AIR%: 18.3%

Paste Savant stats for comparison:
GB% | AIR% | FB% | LD% | PU% | Pull% | Pull AIR%

Bryce Harper - Bats: L
GB%: 40.0%
AIR%: 60.0%
FB%: 31.3%
LD%: 25.8%
PU%: 2.9%
Pull%: 36.4%
Pull AIR%: 12.3%

Paste Savant stats for comparison:
GB% | AIR% | FB% | LD% | PU% | Pull% | Pull AIR%

Freddie Freeman - Bats: L
GB%: 37.4%
AIR%: 62.6%
FB%: 32.9%
LD%: 27.5%
PU%: 2.2%
Pull%: 34.5%
Pull AIR%: 14.5%

Paste Savant stats for comparison:
GB% | AIR% | FB% | LD% | PU% | Pull% | Pull AIR%

Kyle Schwarber - Bats: L
GB%: 43.2%
AIR%: 56.8%
FB%: 26.6%
LD%: 22.4%
PU%: 7.8%
Pull%: 50.5%
Pull AIR%: 21.9%

Paste Savant stats for comparison:
GB% | AIR% | FB% | LD% | PU% | Pull% | Pull AIR%

Francisco Lindor - Bats: L
GB%: 38.3%
AIR%: 61.5%
FB%: 31.8%
LD%: 22.5%
PU%: 7.3%
Pull%: 42.9%
Pull AIR%: 19.6%

Paste Savant stats for comparison:
GB% | AIR% | FB% | LD%